# Run NEO-NN and compare against the full models

Counterpart of `TurbulentTransport/examples/run_TGLFNN.ipynb` for neoclassical
transport: evaluate the NEO-NN surrogates and compare against full NEO
(requires GACODE executables), Hirshman-Sigmar, and Chang-Hinton on the same
plasma. All fluxes are in tgyro-block gyroBohm units, so the numbers are
directly comparable.

In [ ]:
using Pkg
Pkg.activate(; temp=true)
Pkg.develop(; path=dirname(pwd()))   # run from examples/
Pkg.add("Plots")
using NeoclassicalTransport
using NeoclassicalTransport.IMAS
using NeoclassicalTransport.Measurements
using Plots
const NCT = NeoclassicalTransport

In [ ]:
# Load the example plasma (high-betap FPP ODS shipped with the tests) and pick
# mid-radius points. Swap in your own dd here.
dd = IMAS.json2imas(joinpath(dirname(pwd()), "test", "highbetap_fpp_325_ods.json"))
eqt = dd.equilibrium.time_slice[]
cp1d = dd.core_profiles.profiles_1d[]
rho = cp1d.grid.rho_tor_norm
gps = [argmin(abs.(rho .- r)) for r in 0.2:0.05:0.85]
input_neos = [NCT.InputNEO(eqt, cp1d, gp) for gp in gps];

In [ ]:
# Available NEO-NN models (the d3d+mastu+nstx joint nets are the defaults)
NCT.available_models()

In [ ]:
# Run NEO-NN at one radius. warn_nn_train_bounds=true (default) warns whenever
# an input leaves the training space — worth keeping on for a new device.
NCT.run_neonn(input_neos[7])

In [ ]:
# Ensemble uncertainty: mean ± std over the 20 members (Measurements.jl)
NCT.run_neonn(input_neos[7]; uncertain=true, warn_nn_train_bounds=false)

In [ ]:
# Compare with the full models at the same radius
gp = gps[7]
println("rho = ", round(rho[gp]; digits=3), "\n")

sol_nn = NCT.run_neonn(input_neos[7]; warn_nn_train_bounds=false)
println("NEO-NN (joint):    ", sol_nn)

# full NEO — needs a GACODE environment (see README); skipped otherwise.
# On login nodes the `neo -e` wrapper often cannot launch (srun/mpirun dispatch),
# so if a GACODE env is sourced we build (once) and use the serial no-MPI NEO:
if haskey(ENV, "GACODE_ROOT") && !haskey(ENV, "NEO_EXECUTABLE")
    try
        NCT.use_serial_neo!()
        println("using serial NEO: ", ENV["NEO_EXECUTABLE"], "\n")
    catch e
        println("serial NEO build failed — will try the `neo -e` wrapper\n")
    end
end
try
    ineo = deepcopy(input_neos[7])
    # numerics used to generate the NEO-NN training databases
    ineo.COLLISION_MODEL = 4   # full linearized Fokker-Planck
    ineo.N_ENERGY = 12
    ineo.N_THETA = 19
    ineo.N_XI = 19
    sol_neo = NCT.run_neo(ineo)
    println("NEO (FP):          ", sol_neo)
catch e
    println("NEO executable not available — skipping (", sprint(showerror, e)[1:min(end, 60)], ")")
end

pp = NCT.get_plasma_profiles(eqt, cp1d)
eg = NCT.get_equilibrium_geometry(eqt, cp1d)
println("Hirshman-Sigmar:   ", NCT.hirshmansigmar(gp, eqt, cp1d, pp, eg))
println("Chang-Hinton:      ", NCT.changhinton(eqt, cp1d, rho[gp], 1))

Notes on the comparison:
- NEO-NN was trained on **Fokker-Planck** NEO (`COLLISION_MODEL=4`), so it sits
  closer to full NEO than to Hirshman-Sigmar; HS/CH are reduced analytic models.
- The NN uses a 3-species reduction (bulk hydrogenic + lumped impurity +
  electrons); `PARTICLE_FLUX_i` has length 2, while `run_neo`/`hirshmansigmar`
  return one entry per plasma ion.
- The training devices are DIII-D, MAST-U and NSTX with carbon impurity —
  anything else (like this FPP with He+Kr) is an extrapolation; expect the
  dominant channels (`ENERGY_FLUX_i`, `ENERGY_FLUX_e`) to track and the small
  channels to degrade first.

In [ ]:
# Radial profiles: NEO-NN (with ensemble uncertainty band) vs Hirshman-Sigmar
sols_nn = NCT.run_neonn(input_neos; uncertain=true, warn_nn_train_bounds=false)
sols_hs = [NCT.hirshmansigmar(gp, eqt, cp1d, pp, eg) for gp in gps]
x = rho[gps]

plt = plot(; layout=(1, 3), size=(1100, 320), leftmargin=5Plots.mm, bottommargin=5Plots.mm)
for (k, (field, label)) in enumerate([(:ENERGY_FLUX_i, "Qi [GB]"), (:ENERGY_FLUX_e, "Qe [GB]"), (:PARTICLE_FLUX_e, "Γe [GB]")])
    nn = [getfield(s, field) for s in sols_nn]
    plot!(plt[k], x, Measurements.value.(nn); ribbon=Measurements.uncertainty.(nn),
          label="NEO-NN (FP)", lw=2, xlabel="ρ", ylabel=label)
    plot!(plt[k], x, [getfield(s, field) for s in sols_hs]; label="Hirshman-Sigmar", lw=2, ls=:dash)
end
plt

In [ ]:
# Neoclassical flows: poloidal velocities and parallel current from the flow nets
flows = NCT.run_neonn_flow(input_neos; uncertain=true, warn_nn_train_bounds=false)

plt = plot(; layout=(1, 2), size=(900, 320), leftmargin=5Plots.mm, bottommargin=5Plots.mm)
for (name, sel) in [("vpol bulk ion", f -> f.vpol_ion1), ("vpol impurity", f -> f.vpol_ion2), ("vpol electrons", f -> f.vpol_elec)]
    v = sel.(flows)
    plot!(plt[1], x, Measurements.value.(v); ribbon=Measurements.uncertainty.(v), label=name, lw=2)
end
plot!(plt[1]; xlabel="ρ", ylabel="v_pol [v_norm]")
j = [f.jpar for f in flows]
plot!(plt[2], x, Measurements.value.(j); ribbon=Measurements.uncertainty.(j),
      label="⟨j∥·B⟩/B_unit", lw=2, xlabel="ρ", ylabel="jpar [e·n₁·v_norm]", color=:darkred)
plt

In [ ]:
# Single-device nets on the same inputs (24-feature d3d net vs 25-feature joint)
for model in ("neonn_d3d+mastu+nstx_flux", "neonn_d3d_flux", "neonn_mastu+nstx_flux")
    sol = NCT.run_neonn(input_neos[7]; model_filename=model, warn_nn_train_bounds=false)
    println(rpad(model, 40), " Qi = ", round(sol.ENERGY_FLUX_i; sigdigits=4),
            "   Qe = ", round(sol.ENERGY_FLUX_e; sigdigits=4))
end